# Sistema de Recomendação Trocabook

Este notebook tem como objetivo experimentar e validar uma abordagem de
mineração de dados para recomendação personalizada de livros no Trocabook.

O sistema utiliza as interações realizadas pelos usuários na plataforma:

- PESQUISA
- VISUALIZACAO
- INICIO_CONVERSA

As interações serão utilizadas para construir um perfil de interesse do
usuário e recomendar livros com características semelhantes.

## 1. Bibliotecas utilizadas

O experimento utiliza Pandas e NumPy para manipulação dos dados e cálculos
numéricos. Para a representação textual dos livros é utilizado o TF-IDF,
disponibilizado pelo scikit-learn, enquanto a similaridade entre os itens
é calculada por meio da similaridade de cosseno.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Dados utilizados no experimento

Para validar inicialmente a abordagem de recomendação, foram criados dados
simulados de livros e interações de um usuário.

Os livros possuem título, autor e categorias. As interações representam
ações realizadas pelo usuário na plataforma, como pesquisas, visualizações
de anúncios e início de conversas.

Esses dados permitem testar o comportamento do algoritmo antes de sua
integração com os dados reais do Trocabook.

In [2]:
livros = pd.DataFrame([
    {
        "id": "L1",
        "titulo": "Harry Potter e a Pedra Filosofal",
        "autores": "J. K. Rowling",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L2",
        "titulo": "Harry Potter e a Câmara Secreta",
        "autores": "J. K. Rowling",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L3",
        "titulo": "O Hobbit",
        "autores": "J. R. R. Tolkien",
        "categorias": "fantasia aventura"
    },
    {
        "id": "L4",
        "titulo": "Dom Casmurro",
        "autores": "Machado de Assis",
        "categorias": "romance literatura brasileira"
    },
    {
        "id": "L5",
        "titulo": "Clean Code",
        "autores": "Robert Martin",
        "categorias": "programacao tecnologia"
    }
])

livros

,id,titulo,autores,categorias
0,L1,Harry Potter e a Pedra Filosofal,J. K. Rowling,fantasia aventura
1,L2,Harry Potter e a Câmara Secreta,J. K. Rowling,fantasia aventura
2,L3,O Hobbit,J. R. R. Tolkien,fantasia aventura
3,L4,Dom Casmurro,Machado de Assis,romance literatura brasileira
4,L5,Clean Code,Robert Martin,programacao tecnologia


In [3]:
interacoes = pd.DataFrame([
    {
        "uidUsuario": "U1",
        "tipoInteracao": "VISUALIZACAO",
        "uidLivro": "L1",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "INICIO_CONVERSA",
        "uidLivro": "L1",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "VISUALIZACAO",
        "uidLivro": "L3",
        "termoPesquisa": None
    },
    {
        "uidUsuario": "U1",
        "tipoInteracao": "PESQUISA",
        "uidLivro": None,
        "termoPesquisa": "fantasia"
    }
])

interacoes

,uidUsuario,tipoInteracao,uidLivro,termoPesquisa
0,U1,VISUALIZACAO,L1,NaN
1,U1,INICIO_CONVERSA,L1,NaN
2,U1,VISUALIZACAO,L3,NaN
3,U1,PESQUISA,NaN,fantasia


## 3. Ponderação das interações

Nem todas as ações do usuário representam o mesmo nível de interesse.
Por esse motivo, cada tipo de interação recebe um peso diferente:

- PESQUISA: 0.5
- VISUALIZACAO: 1
- INICIO_CONVERSA: 3

Uma pesquisa representa um sinal inicial de interesse, enquanto a
visualização demonstra maior interesse pelo livro. O início de uma
conversa recebe o maior peso por representar uma intenção mais forte
de negociação.

In [4]:
pesos = {
    "PESQUISA": 0.5,
    "VISUALIZACAO": 1,
    "INICIO_CONVERSA": 3
}


In [5]:
interacoes["peso"] = interacoes["tipoInteracao"].map(pesos)

interacoes

,uidUsuario,tipoInteracao,uidLivro,termoPesquisa,peso
0,U1,VISUALIZACAO,L1,NaN,1.0
1,U1,INICIO_CONVERSA,L1,NaN,3.0
2,U1,VISUALIZACAO,L3,NaN,1.0
3,U1,PESQUISA,NaN,fantasia,0.5


## 4. Representação dos livros e cálculo de similaridade

Para identificar livros semelhantes, autores e categorias são transformados
em vetores numéricos utilizando TF-IDF.

Em seguida, é aplicada a similaridade de cosseno para medir o grau de
proximidade entre os livros.

A similaridade final considera duas características:

- categorias: 75%
- autores: 25%

As categorias recebem maior peso porque representam de forma mais ampla
o tipo de conteúdo pelo qual o usuário demonstrou interesse.

Para considerar também as pesquisas realizadas pelo usuário, é criada uma representação textual adicional que combina título, autores e categorias de cada livro. Esse conteúdo é transformado por TF-IDF e utilizado para calcular a similaridade entre o termo pesquisado e os livros disponíveis.

In [6]:
vectorizer_categorias = TfidfVectorizer()
vectorizer_autores = TfidfVectorizer()

matriz_categorias = vectorizer_categorias.fit_transform(
    livros["categorias"]
)

matriz_autores = vectorizer_autores.fit_transform(
    livros["autores"]
)

livros["conteudo_pesquisa"] = (
    livros["titulo"] + " "
    + livros["autores"] + " "
    + livros["categorias"]
)

vectorizer_pesquisa = TfidfVectorizer()

matriz_pesquisa = vectorizer_pesquisa.fit_transform(
    livros["conteudo_pesquisa"]
)

In [7]:
similaridade_categorias = cosine_similarity(matriz_categorias)
similaridade_autores = cosine_similarity(matriz_autores)

In [8]:
import math

PESO_CATEGORIA = 0.75
PESO_AUTOR = 0.25

assert math.isclose(
    PESO_CATEGORIA + PESO_AUTOR,
    1.0
), "A soma dos pesos deve ser igual a 1"

similaridades = (
    similaridade_categorias * PESO_CATEGORIA +
    similaridade_autores * PESO_AUTOR
)

In [9]:
matriz_similaridade = pd.DataFrame(
    similaridades,
    index=livros["titulo"],
    columns=livros["titulo"]
)

matriz_similaridade

titulo,Harry Potter e a Pedra Filosofal,Harry Potter e a Câmara Secreta,O Hobbit,Dom Casmurro,Clean Code
titulo,,,,,
Harry Potter e a Pedra Filosofal,1.00,1.00,0.75,0.0,0.0
Harry Potter e a Câmara Secreta,1.00,1.00,0.75,0.0,0.0
O Hobbit,0.75,0.75,1.00,0.0,0.0
Dom Casmurro,0.00,0.00,0.00,1.0,0.0
Clean Code,0.00,0.00,0.00,0.0,1.0


## 5. Construção do perfil de interesse do usuário

As interações são filtradas pelo usuário e separadas de acordo com sua
finalidade. As interações associadas diretamente a livros são utilizadas
para identificar os itens pelos quais o usuário demonstrou interesse.

Quando existem várias interações com o mesmo livro, seus pesos são somados,
fazendo com que livros com sinais mais fortes de interesse tenham maior
influência na recomendação.

In [10]:
def obter_interacoes_usuario(interacoes, uid_usuario, tipo_interacao=None):
    resultado = interacoes[
        interacoes["uidUsuario"] == uid_usuario
    ]

    if tipo_interacao is not None:
        resultado = resultado[
            resultado["tipoInteracao"] == tipo_interacao
        ]

    return resultado

In [11]:
def obter_interacoes_livros(interacoes_usuario):
    return interacoes_usuario[
        interacoes_usuario["uidLivro"].notna()
    ]

In [12]:
def calcular_interesse_por_livro(interacoes_livros):
    return (
        interacoes_livros
        .groupby("uidLivro")["peso"]
        .sum()
        .reset_index()
    )

## 6. Cálculo dos scores e montagem do ranking

O score de recomendação é construído a partir de duas fontes:

1. similaridade com livros nos quais o usuário já demonstrou interesse;
2. similaridade entre as pesquisas realizadas e o conteúdo dos livros.

Os scores obtidos são somados para produzir uma pontuação final.
Os livros já utilizados nas interações são removidos do resultado e
os demais são ordenados de forma decrescente pelo score.

In [13]:
def calcular_scores_livros(
    livros,
    interesses,
    similaridades
):
    scores = np.zeros(len(livros))

    for _, interesse in interesses.iterrows():
        uid_livro = interesse["uidLivro"]
        peso = interesse["peso"]

        indice = livros.index[
            livros["id"] == uid_livro
        ][0]

        scores += similaridades[indice] * peso

    return scores

In [14]:
def calcular_scores_pesquisas(
    pesquisas,
    livros,
    vectorizer_pesquisa,
    matriz_pesquisa
):
    scores = np.zeros(len(livros))

    for _, pesquisa in pesquisas.iterrows():
        vetor_pesquisa = vectorizer_pesquisa.transform(
            [pesquisa["termoPesquisa"]]
        )

        similaridade = cosine_similarity(
            vetor_pesquisa,
            matriz_pesquisa
        )[0]

        scores += similaridade * pesquisa["peso"]

    return scores

In [15]:
def montar_ranking(livros, scores, livros_interagidos):
    ranking = livros.copy()
    ranking["score"] = scores

    ranking = ranking[
        ~ranking["id"].isin(livros_interagidos)
    ]

    ranking = ranking[
        ranking["score"] > 0
    ]

    return ranking.sort_values(
        by="score",
        ascending=False
    )

## 7. Geração das recomendações

A função a seguir reúne as etapas anteriores em um único fluxo de
recomendação. Ela identifica as interações do usuário, calcula seus
interesses, gera os scores dos livros e das pesquisas e, por fim,
produz o ranking de recomendações.

A última célula executa o experimento para o usuário U1 e apresenta
os livros recomendados juntamente com seus respectivos scores.

In [16]:
def recomendar_livros(
    uid_usuario,
    livros,
    interacoes,
    similaridades,
    vectorizer_pesquisa,
    matriz_pesquisa
):
    interacoes_usuario = obter_interacoes_usuario(
        interacoes,
        uid_usuario
    )

    interacoes_livros = obter_interacoes_livros(
        interacoes_usuario
    )

    pesquisas = obter_interacoes_usuario(
        interacoes,
        uid_usuario,
        "PESQUISA"
    )

    interesses = calcular_interesse_por_livro(
        interacoes_livros
    )

    scores_livros = calcular_scores_livros(
        livros,
        interesses,
        similaridades
    )

    scores_pesquisas = calcular_scores_pesquisas(
        pesquisas,
        livros,
        vectorizer_pesquisa,
        matriz_pesquisa
    )

    scores_finais = scores_livros + scores_pesquisas

    livros_interagidos = interesses["uidLivro"].tolist()

    return montar_ranking(
        livros,
        scores_finais,
        livros_interagidos
    )

In [17]:
recomendacoes = recomendar_livros(
    "U1",
    livros,
    interacoes,
    similaridades,
    vectorizer_pesquisa,
    matriz_pesquisa
)

recomendacoes[["id", "titulo", "score"]]

,id,titulo,score
1,L2,Harry Potter e a Câmara Secreta,4.902054


## 8. Conclusão do experimento

O experimento demonstra que é possível utilizar as interações realizadas
no Trocabook para construir recomendações personalizadas.

A abordagem combina os sinais de interesse do usuário com características
dos livros, utilizando TF-IDF e similaridade de cosseno para identificar
itens relacionados.

Os resultados deste experimento serviram como base para a implementação
do serviço de recomendação utilizado pela aplicação.